# Versionnage et traçage des données

Voyons comment versionner des données et garder la trace des traitements qui les produisent, avec [DVC](https://dvc.org/). DVC sépare le versionnage des données, dont git suit les métadonnées (des fichiers `.dvc`), de leur stockage.

Les étapes sont celles d'un projet réel : versionner les données, les modifier, revenir à une version, décrire leur traitement dans un pipeline, puis tout retrouver sur une autre machine. Pour que la démonstration tourne sans compte, un dépôt git nu sur cette machine, `/tmp/remote.git`, tient lieu de dépôt distant (GitHub, GitLab, DagsHub…), et un dossier, `/tmp/dvc-store`, de stockage distant des données (S3, Google Cloud Storage…).

## Configuration & installations

Cette cellule installe DVC et son extension S3, puis se place dans le dossier du projet, `~/dvc-project`.

In [1]:
!apt-get -qq install tree
!pip install -q --no-warn-script-location dvc dvc-s3
!dvc config --global core.analytics false
!mkdir -p ~/dvc-project
%cd ~/dvc-project

/root/dvc-project


## 1. Créer le dépôt git

Le dépôt git du projet, relié au dépôt « distant ».

In [2]:
!git init --bare -b main /tmp/remote.git
!git init -b main
!git config user.name "Jeanne Durant"
!git config user.email "jeanne@durant.fr"
!git remote add origin /tmp/remote.git

Initialized empty Git repository in /tmp/remote.git/
Initialized empty Git repository in /root/dvc-project/.git/


## 2. Initialiser DVC et son stockage distant

`dvc init` prépare le dépôt, et `dvc remote add -d` déclare le stockage distant par défaut, celui de `dvc push` et `dvc pull` sans `-r`. Avec DagsHub, `dvc remote modify` y ajouterait l'adresse du serveur et, avec `--local`, les identifiants, dans `.dvc/config.local`, que le `.dvc/.gitignore` exclut de git.

In [3]:
!dvc init
!dvc remote add -d origin /tmp/dvc-store

Initialized DVC repository.

You can now commit the changes to git.

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>
Setting 'origin' as a default remote.


In [4]:
!cat .dvc/config .dvc/.gitignore

[core]
    remote = origin
['remote "origin"']
    url = /tmp/dvc-store
/config.local
/tmp
/cache


In [5]:
!git add .dvc
!git commit -m "Initialise DVC"
!git push origin main

[main (root-commit) cfe5e26] Initialise DVC
 3 files changed, 10 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 8 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 533 bytes | 533.00 KiB/s, done.
Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To /tmp/remote.git
 * [new branch]      main -> main


## 3. Récupérer une première version des données

[`dvc get`](https://dvc.org/doc/command-reference/get) télécharge un fichier d'un dépôt git ou DVC, et `dvc add` le fait suivre par DVC : il le copie dans le cache, écrit `data.csv.dvc` et ajoute `data.csv` au `.gitignore`.

In [6]:
!dvc get https://github.com/shuuchuu/datasets wikipedia-movie-plots/wiki_movie_plots_deduped.csv -o data.csv

In [7]:
!dvc add data.csv

⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/root/dvc-project/.dvc/cache/files/md5'| |0/? [00:00<?, 
                                                                                
!
  0%|          |Adding data.csv to cache              0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /root/dvc-project/data.cs0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00,  2.00file/s]

To track the changes with git, run:

	git add .gitignore data.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [8]:
!cat data.csv.dvc
!tree .dvc/cache

outs:
- md5: a546111eeba6497aafc1b3263cf45d6c
  size: 81193310
  hash: md5
  path: data.csv
.dvc/cache
└── files
    └── md5
        └── a5
            └── 46111eeba6497aafc1b3263cf45d6c

4 directories, 1 file


`data.csv.dvc` ne contient que le md5 du fichier, sa taille et son chemin : c'est ce petit fichier que git versionne. Le cache range les données sous leur md5.

## 4. Enregistrer cette version

On commite les fichiers que `dvc add` a nommés, avec l'étiquette `v1`. `git push` n'envoie que ces métadonnées, et `dvc push` les données (81 Mo), depuis le cache, vers le stockage.

In [9]:
!git add data.csv.dvc .gitignore
!git commit -m "Données brutes"
!git tag v1

[main c5e5d61] Données brutes
 2 files changed, 6 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 data.csv.dvc


In [10]:
!git push origin main v1
!dvc push

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 8 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (4/4), 437 bytes | 437.00 KiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To /tmp/remote.git
   cfe5e26..c5e5d61  main -> main
 * [new tag]         v1 -> v1
Pushing
!
  0% Checking cache in '/tmp/dvc-store/files/md5'|   |0/? [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/root/dvc-project/.dvc/cache/files/md5'| |0/? [00:00<?, 
                                                                                
!
  0%|          |Pushing to local                      0/1 [00:00<?,     ?file/s]
Pushing
1 file pushed


## 5. Modifier les données

On trie les lignes de `data.csv` dans l'ordre inverse, puis on enregistre cette nouvelle version comme la première, avec l'étiquette `v2`.

In [11]:
!sort -r data.csv -o data.csv

In [12]:
!dvc add data.csv

⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/root/dvc-project/.dvc/cache/files/md5'| |0/? [00:00<?, 
                                                                                
!
  0%|          |Adding data.csv to cache              0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /root/dvc-project/data.cs0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00,  2.30file/s]

To track the changes with git, run:

	git add data.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [13]:
!cat data.csv.dvc
!tree .dvc/cache

outs:
- md5: 06d279899c8c10d4667a551b39950925
  size: 81193310
  hash: md5
  path: data.csv
.dvc/cache
└── files
    └── md5
        ├── 06
        │   └── d279899c8c10d4667a551b39950925
        └── a5
            └── 46111eeba6497aafc1b3263cf45d6c

5 directories, 2 files


In [14]:
!git add data.csv.dvc
!git commit -m "Données v2"
!git tag v2

[main 8e4ff18] Données v2
 1 file changed, 1 insertion(+), 1 deletion(-)


In [15]:
!git push origin main v2
!dvc push

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 8 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 338 bytes | 84.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To /tmp/remote.git
   c5e5d61..8e4ff18  main -> main
 * [new tag]         v2 -> v2
Pushing
!
  0% Checking cache in '/tmp/dvc-store/files/md5'|   |0/? [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/root/dvc-project/.dvc/cache/files/md5'| |0/? [00:00<?, 
                                                                                
!
  0%|          |Pushing to local                      0/1 [00:00<?,     ?file/s]
Pushing
1 file pushed


Le md5 a changé, pas la taille. Le cache contient maintenant les deux versions, chacune sous son md5.

## 6. Revenir aux données originales

La nouvelle version pose problème : on revient à la première. `git restore --source v1` remet le `data.csv.dvc` de `v1`, et `dvc checkout` les données qu'il désigne, depuis le cache.

In [16]:
!git restore --source v1 data.csv.dvc
!dvc checkout

Building workspace index                              |1.00 [00:00, 32.4entry/s]
Comparing indexes                                     |2.00 [00:00,  701entry/s]
Applying changes                                      |1.00 [00:00,  8.84file/s]
M       data.csv


In [17]:
!cat data.csv.dvc

outs:
- md5: a546111eeba6497aafc1b3263cf45d6c
  size: 81193310
  hash: md5
  path: data.csv


In [18]:
!git add data.csv.dvc
!git commit -m "Retour aux données v1"
!git tag v3

[main 8546a4d] Retour aux données v1
 1 file changed, 1 insertion(+), 1 deletion(-)


In [19]:
!git push origin main v3
!dvc push

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 8 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 347 bytes | 347.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To /tmp/remote.git
   8e4ff18..8546a4d  main -> main
 * [new tag]         v3 -> v3
Pushing
!
  0% Checking cache in '/tmp/dvc-store/files/md5'|   |0/? [00:00<?,    ?files/s]
Pushing
Everything is up to date.


`dvc push` n'a rien à envoyer : les données de `v1` sont déjà sur le stockage, sous le même md5.

## 7. Une étape de pipeline

Le script `upper.py` recopie un fichier en majuscules. `dvc stage add` en fait une étape de pipeline, avec ses dépendances (le script, les données) et sa sortie, et `dvc repro` l'exécute et écrit `dvc.lock`.

In [20]:
%%writefile upper.py
from pathlib import Path
from sys import argv

texte = Path(argv[1]).read_text(encoding="utf-8")
Path(argv[2]).write_text(texte.upper(), encoding="utf-8")

Writing upper.py


In [21]:
!dvc stage add --name uppercase --deps upper.py --deps data.csv --outs data-upper.csv python upper.py data.csv data-upper.csv

Added stage 'uppercase' in 'dvc.yaml'

To track the changes with git, run:

	git add dvc.yaml .gitignore

To enable auto staging, run:

	dvc config core.autostage true


In [22]:
!dvc repro

'data.csv.dvc' didn't change, skipping
Running stage 'uppercase':
> python upper.py data.csv data-upper.csv
Generating lock file 'dvc.lock'
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


In [23]:
!git add upper.py dvc.yaml dvc.lock .gitignore
!git commit -m "Pipeline en majuscules"
!git push origin main
!dvc push

[main 729ae2b] Pipeline en majuscules
 4 files changed, 32 insertions(+)
 create mode 100644 dvc.lock
 create mode 100644 dvc.yaml
 create mode 100644 upper.py
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 8 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 869 bytes | 869.00 KiB/s, done.
Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To /tmp/remote.git
   8546a4d..729ae2b  main -> main
Pushing
!
  0% Checking cache in '/tmp/dvc-store/files/md5'|   |0/? [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/root/dvc-project/.dvc/cache/files/md5'| |0/? [00:00<?, 
                                                                                
!
  0%|          |Pushing to local                      0/1 [00:00<?,     ?file/s]
Pushing
1 file pushed


In [24]:
!dvc repro
!cat dvc.lock

'data.csv.dvc' didn't change, skipping
Stage 'uppercase' didn't change, skipping
Data and pipelines are up to date.
schema: '2.0'
stages:
  uppercase:
    cmd: python upper.py data.csv data-upper.csv
    deps:
    - path: data.csv
      hash: md5
      md5: a546111eeba6497aafc1b3263cf45d6c
      size: 81193310
    - path: upper.py
      hash: md5
      md5: 30c7765868017cad06d38dc2caa5a09c
      size: 155
    outs:
    - path: data-upper.csv
      hash: md5
      md5: 2a6ce2d726d43441b0b12c6e3d35c7a1
      size: 81059022


Le second `dvc repro` saute l'étape : rien n'a changé depuis les md5 de `dvc.lock`. `dvc.lock` dit exactement à partir de quoi et comment `data-upper.csv` a été produit : c'est le traçage des données.

## 8. Sur une autre machine

Une collègue rejoint le projet : elle clone le dépôt et récupère les données avec `dvc pull`. Avec DagsHub, elle redonnerait d'abord au clone ses identifiants, que git n'a pas transmis.

In [25]:
!git clone /tmp/remote.git ~/other-machine

Cloning into '/root/other-machine'...
done.


In [26]:
%cd ~/other-machine

/root/other-machine


In [27]:
!dvc pull

Fetching
!
  0% Checking cache in '/root/other-machine/.dvc/cache/files/md5'| |0/? [00:00<?
                                                                                
!
  0% Checking cache in '/tmp/dvc-store/files/md5'|   |0/? [00:00<?,    ?files/s]
                                                                                
!
  0%|          |Fetching from local                   0/2 [00:00<?,     ?file/s]
100%|██████████|Fetching from local               2/2 [00:00<00:00, 11.41file/s]
Fetching
Building workspace index                              |0.00 [00:00,    ?entry/s]
Comparing indexes                                     |3.00 [00:00,  468entry/s]
Applying changes                                      |2.00 [00:00,  9.65file/s]
A       data-upper.csv
A       data.csv
2 files fetched and 2 files added


In [28]:
!ls
!dvc status
!dvc repro

data-upper.csv	data.csv  data.csv.dvc	dvc.lock  dvc.yaml  upper.py
Data and pipelines are up to date.
'data.csv.dvc' didn't change, skipping
Stage 'uppercase' didn't change, skipping
Data and pipelines are up to date.


Le clone a `data.csv` (dans sa version de `v3`) et `data-upper.csv`, sans avoir réexécuté le script : les données et la pipeline sont à jour.